In [4]:
######## check data ########
import pandas as pd
import os
pd.set_option('display.max_colwidth', 1000)
 
data_path = 'out/pq_0423/'
files = os.listdir(data_path)

files_path = [os.path.join(data_path, f) for f in files]

for file_path in files_path:
    df = pd.read_parquet(file_path)
    print(f"{file_path} data shape: {df.shape}")
    print(f"{'-'*80}")
    print(df.head(10))
    print("\n\n")
    

out/pq_0423/accounts.parquet data shape: (200, 5)
--------------------------------------------------------------------------------
             account_id            customer_id account_type   balance  \
0  account_bada0c941679  customer_0e2f8958de43     checking   9828.38   
1  account_a708728034ee  customer_8186a57611a7      savings   8779.91   
2  account_eac7dc2e9099  customer_2defe1935c62   investment   3153.15   
3  account_51958000c3f3  customer_54cc1e2a5e4c       credit   1492.33   
4  account_21acfdb51b55  customer_f1f8343ea99f      savings  14617.37   
5  account_aff7ff5d3b1d  customer_dc713d960c0f   investment  15544.56   
6  account_0840323439ed  customer_e4a4474c9b33     checking    765.69   
7  account_7521aa18dde4  customer_11906f503488     checking   2429.90   
8  account_74f76d09739c  customer_3ae8cc938dcd   investment  10936.01   
9  account_e267e12b65e5  customer_626d719d81cf     checking   2820.31   

                                                                 

In [13]:
import os
from datetime import date, datetime, timezone

import pandas as pd

data_path = "out/pq_0423/"  

def pq(name: str) -> str:
    root = data_path.rstrip("/\\") + os.sep
    return root + name + ".parquet"


df = pd.read_parquet(pq("client_accounts"))
cov = pd.read_parquet(pq("coverage_assignments"))

# ---------- 1) RM / 管户行：重复键 ----------
df[df.duplicated(["serving_account_id", "customer_id"], keep=False)]
df[df.duplicated(["serving_account_id"], keep=False)]  

# ---------- 2) Active：必须有主责；每人一条 Primary ----------
active = df["account_status"].astype(str).str.strip() == "Active"
df[active & df["coverage_owner_id"].isna()]

prim = cov[cov["assignment_type"].astype(str).str.strip() == "Primary"]
n_prim = prim.groupby(prim["serving_account_id"].astype(str), dropna=False).size()
# Active 的 serving_account_id 应满足 n_prim == 1
act_sid = set(df.loc[active, "serving_account_id"].astype(str))
[(s, n_prim.get(s, 0)) for s in sorted(act_sid) if n_prim.get(s, 0) != 1]

# # ---------- 3) is_servicing_eligible 与 KYC/到期/制裁 是否一致 ----------
raw = os.environ.get("FDS_AS_OF_DATE", "").strip()
as_of = date(*[int(x) for x in raw[:10].split("-")]) if raw and len(raw) >= 10 else datetime.now(timezone.utc).date()
kexp = pd.to_datetime(df["kyc_expiry_date"], utc=True, errors="coerce").dt.date
expected = (
    (df["kyc_status"].astype(str).str.strip() == "Active")
    & (df["sanctions_status"].astype(str).str.strip() == "Clear")
    & (kexp >= as_of)
)
# # 若 kyc_expiry 解析失败，上面可能需显式把 NaT 判为不可服务
mismatch = df[expected != df["is_servicing_eligible"].astype(bool)]
mismatch

,serving_account_id,customer_id,account_status,kyc_status,kyc_expiry_date,sanctions_status,coverage_owner_id,is_servicing_eligible,account_metadata_json
